# Transformer-based NER

In [1]:
# ! pip install torch --index-url https://download.pytorch.org/whl/cu118

In [2]:
# ! pip install -U spacy[transformers]

In [3]:
# ! python -m spacy init config config_tr.cfg --lang en --pipeline transformer,ner --optimize efficiency --force -G

In [4]:
! python -m spacy train config_tr.cfg --output ./transformer --paths.train ./train_data.spacy --paths.dev ./val_data.spacy --gpu-id 0

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Saving to output directory: transformer
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/y

In [5]:
import spacy, time
from spacy.training import Corpus
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

nlp_ner = spacy.load("./transformer/model-best")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/o Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    # Use ['ents_p']
print(f"Recall:    {scores['ents_r']:.3f}")    # Use ['ents_r']
print(f"F1-Score:  {scores['ents_f']:.3f}")    # Use ['ents_f']

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['transformer', 'ner']
=== Tok2Vec w/o Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.833
Recall:    0.658
F1-Score:  0.735


In [6]:
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./transformer/model-best")
    
    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 1.52 seconds
Memory used: 339.99 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE


In [7]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import glob
test_file = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
file_idx = 1

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [8]:
nlp_ner = spacy.load("./transformer/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [9]:
nlp_ner = spacy.load("./transformer/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/ Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

['transformer', 'ner', 'entity_ruler']
=== Tok2Vec w/ Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.810
Recall:    0.757
F1-Score:  0.782


In [10]:
from memory_profiler import memory_usage

def test_inference(text: str):
    
    nlp_ner = spacy.load("./transformer/model-best")
    ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
    )
    ruler.from_disk("entity_rulers.jsonl")

    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")

    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 1.61 seconds
Memory used: 749.73 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding point C1 : DESTINATION


# Train with entity rule spans

In [11]:
! python -m spacy train config_tr.cfg \
    --output ./transformer_rules \
        --paths.train ./augmented_train_data.spacy \
            --paths.dev ./val_data.spacy \
                --gpu-id 0

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Saving to output directory: transformer_rules
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/

In [12]:
from spacy.training import Corpus

nlp_ner = spacy.load("./transformer_rules/model-best")

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===
=== NER Evaluation Metrics ===
Precision: 0.850
Recall:    0.743
F1-Score:  0.793


In [13]:
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./transformer/model-best")
    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 1.44 seconds
Memory used: 699.08 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE


In [14]:
nlp_ner = spacy.load("./transformer_rules/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")

['transformer', 'ner', 'entity_ruler']
=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===
=== NER Evaluation Metrics ===
Precision: 0.858
Recall:    0.836
F1-Score:  0.847


In [15]:
import spacy
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./transformer/model-best")
    ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
    )
    ruler.from_disk("entity_rulers.jsonl")

    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 1.52 seconds
Memory used: 419.77 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding point C1 : DESTINATION
